In [1]:
# ===== CELL 1 / SETUP -- run BEFORE the two-stage cell (arguswala.py) =====
# Provides: meta, AUD, support, target_pool, decoy_pools, calib_beds, eval_beds,
#           to32k, perch_embed, perch_embed_ctx, perch_clf, l2n, rms
#
# THREE CORRECTNESS FIXES over the previous version:
#  (a) NO LEAKAGE. perch_clf is calibrated on `support` ONLY (the 5 labelled
#      events). target_pool is now purely held-out evaluation data. Previously
#      target_pool was in the training set AND was the pool the evaluation
#      planted calls from -- the verifier had memorised the test calls.
#  (b) REAL 5s CONTEXT. Perch is a 5s/32kHz model. Feeding it a 1s call plus 4s
#      of trailing digital silence is 80% zeros and not a distribution it was
#      trained on. perch_embed_ctx() cuts a real 5s window out of the recording
#      around the event; isolated clips fall back to CENTRED padding (Perch's
#      documented convention) rather than trailing.
#  (c) DOMAIN-MATCHED CALIBRATION. perch_clf is fit on calls injected into real
#      soundscape beds at the evaluation SNRs, with windows extracted exactly the
#      way two_stage() extracts them at inference. Previously it trained on clean
#      Xeno-Canto peaks and was applied to noisy soundscape events.
import os, glob
import numpy as np
import pandas as pd
import librosa
import tensorflow_hub as hub
from sklearn.linear_model import LogisticRegression

SR = 22050
SEG_SEC = 1.0
CLEN = int(SEG_SEC * SR)
TARGET = "whbsho3"        # White-bellied Sholakili -- same target as argus.py
N_SHOT = 5
N_DECOYS = 4
N_CANDIDATES = 30         # non-target species considered for decoy ranking
CALIB_PER_SPECIES = 6     # recordings per candidate species (ranking + negatives)
DECOY_RECORDINGS = 20     # deeper pool for the 4 CHOSEN decoys, so the specificity
                          # control isn't replaying the same 6 clips over and over
N_CALIB_BEDS, N_EVAL_BEDS = 4, 10   # disjoint: no bed-level leakage either
SNR_LEVELS = (9, 6, 3, 0, -3, -6)

root = os.path.dirname(glob.glob('/kaggle/input/**/train_metadata.csv', recursive=True)[0])
AUD = os.path.join(root, 'train_audio')
SS = os.path.join(root, 'unlabeled_soundscapes')
meta = pd.read_csv(os.path.join(root, 'train_metadata.csv'))

# ------------------------------------------------------------ audio helpers
def rms(x):
    return float(np.sqrt(np.mean(np.square(x, dtype=np.float64)) + 1e-12))

def ld(p, dur=None):
    y, _ = librosa.load(p, sr=SR, mono=True, duration=dur)
    return y.astype('float32')

def l2n(x):
    x = np.asarray(x, dtype="float32")
    return x / (np.linalg.norm(x, axis=-1, keepdims=True) + 1e-9)

def cos(a, b):
    return float(a @ b / ((np.linalg.norm(a) + 1e-9) * (np.linalg.norm(b) + 1e-9)))

def loudest_offset(y, n=CLEN):
    if len(y) <= n: return 0
    e = np.square(y, dtype=np.float64); cs = np.concatenate([[0.0], np.cumsum(e)])
    return int(np.argmax(cs[n:] - cs[:-n]))

def peak_windows(y, k=1, n=CLEN):
    """Top-k non-overlapping highest-energy windows of length n."""
    if len(y) <= n: return [np.pad(y, (0, n - len(y))).astype('float32')]
    e = np.square(y, dtype=np.float64); cs = np.concatenate([[0.0], np.cumsum(e)])
    w = cs[n:] - cs[:-n]
    out, taken = [], []
    for _ in range(k):
        pick = next((int(i) for i in np.argsort(w)[::-1] if all(abs(int(i) - t) >= n for t in taken)), None)
        if pick is None: break
        taken.append(pick); out.append(y[pick:pick + n].astype('float32'))
    return out

def build_support(recordings, shifts=(-0.25, 0.0, 0.25)):
    out = []
    for y in recordings:
        base = loudest_offset(y)
        for shift in shifts:
            a = int(np.clip(base + shift * SR, 0, max(0, len(y) - CLEN)))
            w = y[a:a + CLEN]
            if len(w) < CLEN: w = np.pad(w, (0, CLEN - len(w)))
            out.append(w.astype('float32'))
    return out

def place(y, pool, rng, spans, snr_db, gap_s=0.4):
    """Mix one random call from `pool` into `y` at a free position. Returns its
    centre in seconds, or None if no non-overlapping slot was found."""
    for _ in range(80):
        pos = int(rng.integers(0, len(y) - CLEN))
        if any(pos < e + int(gap_s*SR) and pos + CLEN > s - int(gap_s*SR) for s, e in spans):
            continue
        c = pool[int(rng.integers(len(pool)))][:CLEN]
        if len(c) < CLEN: c = np.pad(c, (0, CLEN - len(c)))
        g = rms(y[pos:pos+CLEN]) * (10 ** (float(snr_db)/20)) / (rms(c) + 1e-9)
        y[pos:pos+CLEN] += c.astype('float32') * g
        spans.append((pos, pos + CLEN))
        return (pos + CLEN/2) / SR
    return None

# ------------------------------------------------------------ Perch
def _find_local_perch():
    hits = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/saved_model.pb', recursive=True)
            if 'bird-vocalization' in p.lower() or 'perch' in p.lower()]
    if not hits: return None
    def _ver(p):
        try: return int(os.path.basename(p))
        except ValueError: return -1
    return max(hits, key=_ver)   # glob() order isn't guaranteed; pick the highest version deterministically

PERCH_HANDLE = _find_local_perch() or "https://www.kaggle.com/models/google/bird-vocalization-classifier/tensorFlow2/bird-vocalization-classifier/4"
print("Perch source:", PERCH_HANDLE)
perch = hub.load(PERCH_HANDLE)
PERCH_SR, PERCH_WIN = 32000, 5.0     # Perch expects 5s @ 32kHz
PERCH_LEN = int(PERCH_WIN * PERCH_SR)
PERCH_BATCH = 16
_perch_batched = True                # flipped off automatically if batching fails

def to32k(y):
    return librosa.resample(np.asarray(y, dtype="float32"), orig_sr=SR, target_sr=PERCH_SR)

def _infer(batch):
    """v8+ returns a dict of heads (label/genus/family/order/frontend/embedding).
    v1-v4 return a plain (logits, embeddings) 2-tuple -- no 'embedding' key at all."""
    out = perch.infer_tf(batch)
    return (out["embedding"] if isinstance(out, dict) else out[1]).numpy()

def _perch_run(waves):
    """waves: (n, PERCH_LEN) float32 -> (n, D). Fixed batch size avoids TF retracing;
    falls back to per-clip inference if this model version rejects a batch."""
    global _perch_batched
    n = len(waves)
    if n == 0: return np.zeros((0, 1280), dtype="float32")
    if _perch_batched:
        try:
            out = []
            for i in range(0, n, PERCH_BATCH):
                chunk = waves[i:i+PERCH_BATCH]; k = len(chunk)
                if k < PERCH_BATCH:
                    chunk = np.concatenate([chunk, np.zeros((PERCH_BATCH-k, PERCH_LEN), dtype="float32")])
                out.append(_infer(chunk)[:k])
            return np.concatenate(out)
        except Exception as ex:
            print("batched Perch inference failed, falling back to per-clip:", type(ex).__name__)
            _perch_batched = False
    return np.stack([_infer(w[np.newaxis, :])[0] for w in waves])

def perch_embed(waveforms):
    """Isolated clips (no surrounding audio available) -> (n, D).
    Uses CENTRED zero-padding, Perch's documented convention for short examples."""
    ws = []
    for w in waveforms:
        w = to32k(w)
        if len(w) >= PERCH_LEN:
            w = w[:PERCH_LEN]
        else:
            pad = PERCH_LEN - len(w); lo = pad // 2
            w = np.pad(w, (lo, pad - lo))
        ws.append(w.astype("float32"))
    return _perch_run(np.stack(ws)) if ws else np.zeros((0, 1280), dtype="float32")

def perch_embed_ctx(y32, centres_sec):
    """5s windows cut from a 32kHz recording, centred on each event time -- real
    acoustic context, which is what Perch is built for. Edge windows are clamped
    inside the recording; only a recording shorter than 5s gets centred padding."""
    ws = []
    for c in centres_sec:
        a = int(round(c * PERCH_SR)) - PERCH_LEN // 2
        a = max(0, min(a, max(0, len(y32) - PERCH_LEN)))
        w = y32[a:a + PERCH_LEN]
        if len(w) < PERCH_LEN:
            pad = PERCH_LEN - len(w); lo = pad // 2
            w = np.pad(w, (lo, pad - lo))
        ws.append(w.astype("float32"))
    return _perch_run(np.stack(ws)) if ws else np.zeros((0, 1280), dtype="float32")

# ------------------------------------------- support / held-out target calls
target_rows = meta[meta.primary_label == TARGET].sort_values('rating', ascending=False)
files = target_rows['filename'].tolist()
if len(files) < N_SHOT + 2:
    raise ValueError(f"not enough recordings for {TARGET}: found {len(files)}")

enrol_raw = [ld(os.path.join(AUD, f), 30) for f in files[:N_SHOT]]
support = build_support(enrol_raw)
# EVALUATION ONLY -- never seen by perch_clf.
target_pool = [w for f in files[N_SHOT:] for w in peak_windows(ld(os.path.join(AUD, f), 30), k=1)]

PROTO = l2n(perch_embed(support)).mean(0); PROTO /= np.linalg.norm(PROTO) + 1e-9
print(f"target={TARGET}: {len(support)} support views (calibration), "
      f"{len(target_pool)} held-out calls (evaluation only)")

# ------------------------------- candidate species -> decoys by Perch distance
candidate_species = (meta[meta.primary_label != TARGET]['primary_label']
                     .value_counts().head(N_CANDIDATES).index.tolist())
species_calls, species_emb = {}, {}
for sp in candidate_species:
    calls = []
    for f in meta[meta.primary_label == sp]['filename'].tolist()[:CALIB_PER_SPECIES]:
        try: calls += peak_windows(ld(os.path.join(AUD, f), 30), k=1)
        except Exception: pass
    if not calls: continue
    species_calls[sp] = calls
    species_emb[sp] = l2n(perch_embed(calls)).mean(0)
print(f"embedded {len(species_calls)}/{len(candidate_species)} candidate species")

ranked = sorted(species_emb, key=lambda sp: -cos(species_emb[sp], PROTO))
DECOYS = ranked[:N_DECOYS]
calib_neg_species = [sp for sp in ranked if sp not in DECOYS]   # held out from decoys

# deeper pools for the chosen decoys so the control isn't replaying 6 clips
decoy_pools = {}
for sp in DECOYS:
    pool = []
    for f in meta[meta.primary_label == sp]['filename'].tolist()[:DECOY_RECORDINGS]:
        try: pool += peak_windows(ld(os.path.join(AUD, f), 30), k=1)
        except Exception: pass
    decoy_pools[sp] = pool or species_calls[sp]
print("decoy species (closest to target in Perch space):")
for sp in DECOYS:
    print(f"  {sp}: sim={cos(species_emb[sp], PROTO):.3f}  ({len(decoy_pools[sp])} calls)")

# ------------------------------------------------------------ soundscape beds
bed_paths = sorted(glob.glob(os.path.join(SS, "*.ogg")))[:N_CALIB_BEDS + N_EVAL_BEDS]
all_beds = [ld(p, 240.0) for p in bed_paths]
calib_beds, eval_beds = all_beds[:N_CALIB_BEDS], all_beds[N_CALIB_BEDS:]
print(f"beds: {len(calib_beds)} calibration + {len(eval_beds)} evaluation (disjoint)")

# ------------------------------------------- domain-matched perch_clf training
# Positives : the 5 labelled support events, injected into real beds at the
#             evaluation SNRs, with centre jitter (detected centres are never exact).
# Negatives : other-species calls injected the same way, PLUS plain background
#             windows -- stage-1 fires on background too, not only on other birds.
neg_calls = [w for sp in calib_neg_species for w in species_calls[sp]]
rng_cal = np.random.default_rng(11)
X_parts, y_all = [], []
for bed in calib_beds:
    y_bed = bed.copy(); spans, centres, labels = [], [], []
    plan = [(1, support)] * 12 + [(0, neg_calls)] * 12
    plan = [plan[i] for i in rng_cal.permutation(len(plan))]   # interleaves labels across SNRs
    for i, (lab, pool) in enumerate(plan):
        c = place(y_bed, pool, rng_cal, spans, SNR_LEVELS[i % len(SNR_LEVELS)])
        if c is None: continue
        centres.append(c + float(rng_cal.uniform(-0.25, 0.25))); labels.append(lab)
    for _ in range(8):   # background-only negatives
        pos = int(rng_cal.integers(0, len(y_bed) - CLEN))
        if any(pos < e and pos + CLEN > s for s, e in spans): continue
        centres.append((pos + CLEN/2) / SR); labels.append(0)
    X_parts.append(perch_embed_ctx(to32k(y_bed), centres)); y_all += labels

X = l2n(np.concatenate(X_parts)); y = np.array(y_all)
# C is deliberately small: 1280-d embeddings with ~100 samples separate perfectly at
# the default C=1, which saturates predict_proba to 0/1 and turns v into a hard veto.
# Softer probabilities rank better, which is what AP and the fusion rules need.
CLF_C = 0.1
perch_clf = LogisticRegression(C=CLF_C, max_iter=2000, class_weight='balanced').fit(X, y)
print(f"perch_clf: {int((y==1).sum())} target / {int((y==0).sum())} non-target windows "
      f"from {len(calib_beds)} beds ({len(calib_neg_species)} negative species) | "
      f"train acc {perch_clf.score(X, y):.2f}")


/usr/local/lib/python3.12/dist-packages/tensorflow_hub/__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


Perch source: /kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/bird-vocalization-classifier/8


I0000 00:00:1785548902.235778      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1785548902.238843      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
2026-08-01 01:48:53.077618: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-08-01 01:48:53.221734: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-08-01 01:48:53.428529: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:

target=whbsho3: 15 support views (calibration), 17 held-out calls (evaluation only)
embedded 30/30 candidate species
decoy species (closest to target in Perch space):
  blrwar1: sim=0.755  (20 calls)
  commoo3: sim=0.640  (20 calls)
  litgre1: sim=0.623  (20 calls)
  grtdro1: sim=0.616  (20 calls)
beds: 4 calibration + 10 evaluation (disjoint)
perch_clf: 48 target / 74 non-target windows from 4 beds (26 negative species) | train acc 0.93


In [ ]:
# ===== CELL 2 / TWO-STAGE: ARGUS proposes, Perch verifies =====
# Needs from CELL 1 (arguswala_setup.py): support, target_pool, decoy_pools, meta,
#   AUD, eval_beds, to32k, perch_embed_ctx, perch_clf, l2n, SNR_LEVELS
#
# Optimises F1 subject to keeping the specificity ratio low. Every candidate event
# is detected ONCE and cached with both scores -- stage-1's probability p and Perch's
# verification probability v -- then each fusion rule is scored from that same cache.
# The comparison is therefore PAIRED: identical events, identical planting, only the
# scoring differs, so differences between rules are not seed noise.
import numpy as np, os, glob, copy, librosa, torch, torch.nn as nn, torch.optim as optim

SR, N_FFT, HOP, N_MELS = 22050, 1024, 256, 128
SEG_SEC = 1.0; SEG_FR = int(np.ceil(SEG_SEC*SR/HOP)); CLEN = int(SEG_SEC*SR); EMB = 128
FT_STEPS, FT_LR, FT_BATCH = 250, 1e-4, 16
N_INJECT, N_NEG, INJ_SNR = 45, 200, (-6, 12)
CAND_FLOOR, STRIDE, IOU_THR = 0.10, max(1, SEG_FR//3), 0.30
MAX_EVENT_SEC = 10.0
CALLS_PER_BED = 12
device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); print("device:", device)

def rms(x): return float(np.sqrt(np.mean(np.square(x, dtype=np.float64)) + 1e-12))
def ld(p, dur=None):
    y, _ = librosa.load(p, sr=SR, mono=True, duration=dur); return y.astype("float32")
def pcen(y):
    m = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS, power=1.0)
    return librosa.pcen(m*(2**31), sr=SR, hop_length=HOP).astype("float32")
def crop(f, c):
    a = c - SEG_FR//2; p = np.full((N_MELS, SEG_FR), f.min(), dtype="float32")
    lo, hi = max(0, a), min(f.shape[1], a+SEG_FR)
    if hi > lo: p[:, (lo-a):(lo-a)+(hi-lo)] = f[:, lo:hi]
    return p
def loud_off(y, n=CLEN):
    if len(y) <= n: return 0
    e = np.square(y, dtype=np.float64); cs = np.concatenate([[0.0], np.cumsum(e)])
    return int(np.argmax(cs[n:]-cs[:-n]))
def specaug(x, rng):
    if rng.random() >= 0.6: return x
    x = x.copy(); fl = x.min()
    for _ in range(2):
        k = int(rng.integers(0, 21))
        if k: a = int(rng.integers(0, max(1, N_MELS-k))); x[a:a+k, :] = fl
        k = int(rng.integers(0, 11))
        if k: a = int(rng.integers(0, max(1, SEG_FR-k))); x[:, a:a+k] = fl
    return x
def inject(bed, calls, n, snr, rng, gap_s=0.4):
    y = bed.astype("float32").copy(); L = len(y); gap = int(gap_s*SR); sp = []
    if L <= CLEN: return y, sp
    for _ in range(n*60):
        if len(sp) >= n: break
        pos = int(rng.integers(0, L-CLEN))
        if any(pos < e+gap and pos+CLEN > s-gap for s, e in sp): continue
        c = calls[int(rng.integers(len(calls)))][:CLEN]
        if len(c) < CLEN: c = np.pad(c, (0, CLEN-len(c)))
        g = rms(y[pos:pos+CLEN])*(10**(float(rng.uniform(*snr))/20))/(rms(c)+1e-9)
        y[pos:pos+CLEN] += c.astype("float32")*g; sp.append((pos, pos+CLEN))
    return y, sorted(sp)
def iou(a, b):
    it = max(0.0, min(a[1], b[1]) - max(a[0], b[0]))
    u = (a[1]-a[0]) + (b[1]-b[0]) - it
    return it/u if u > 0 else 0.0

class Enc(nn.Module):
    def __init__(s, emb=EMB):
        super().__init__()
        def B(i, o): return nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False),
                                          nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))
        s.e = nn.Sequential(B(1,128), B(128,128), B(128,128), B(128,emb)); s.p = nn.AdaptiveAvgPool2d(1)
    def forward(s, x): return s.p(s.e(x.unsqueeze(1))).flatten(1)

def train_enc(bank, rng, episodes=1000, nway=6, k=4, q=4):
    m = Enc().to(device); opt = optim.Adam(m.parameters(), 1e-3); ce = nn.CrossEntropyLoss()
    cl = list(bank); m.train()
    for ep in range(episodes):
        ch = rng.choice(cl, size=min(nway, len(cl)), replace=False)
        sx, sy, qx, qy = [], [], [], []
        for lab, c in enumerate(ch):
            idx = rng.permutation(len(bank[c]))[:k+q]
            for j, i in enumerate(idx):
                (sx if j < k else qx).append(torch.tensor(specaug(bank[c][i], rng)))
                (sy if j < k else qy).append(lab)
        sx = torch.stack(sx).to(device); qx = torch.stack(qx).to(device)
        sy = torch.tensor(sy).to(device); qy = torch.tensor(qy).to(device)
        se, qe = m(sx), m(qx)
        pr = torch.stack([se[sy == c].mean(0) for c in torch.unique(sy)])
        loss = ce(-torch.cdist(qe, pr), qy); opt.zero_grad(); loss.backward(); opt.step()
        if (ep+1) % 250 == 0: print(f"    enc ep {ep+1}/{episodes} loss {loss.item():.3f}")
    return m

def stitch(frames, probs):
    ab = probs >= CAND_FLOOR; ev = []; i = 0
    while i < len(frames):
        if not ab[i]: i += 1; continue
        j = i
        while j+1 < len(frames) and ab[j+1] and frames[j+1]-frames[j] <= STRIDE*1.5: j += 1
        pk = i + int(np.argmax(probs[i:j+1]))
        s = max(0.0, frames[i]*HOP/SR - SEG_SEC/2); e = frames[j]*HOP/SR + SEG_SEC/2
        if e - s > MAX_EVENT_SEC:
            c = frames[pk]*HOP/SR; s, e = max(0.0, c-SEG_SEC/2), c+SEG_SEC/2
        ev.append((s, e, float(probs[pk]))); i = j+1
    return ev

def stage1(enc, rec, sup, rng):
    # N_INJECT=45 was tuned on 240 s beds (~19% of the recording covered by injected
    # calls). Using that count on a short clip both fails to place them and leaves no
    # clean background to sample negatives from. Hold the DENSITY constant instead --
    # this is identity for 240 s recordings, so research numbers are unchanged.
    n_inj = max(8, int(round(N_INJECT * len(rec) / (240.0 * SR))))
    aw, sp = inject(rec, sup, n_inj, INJ_SNR, rng)
    af, sf = pcen(aw), pcen(rec)
    pc = [((a+b)//2)//HOP for a, b in sp]
    if not pc: return []
    pos = [crop(af, c) for c in pc]
    g = SEG_FR; hi = af.shape[1]-g; neg = []
    for _ in range(N_NEG*40):
        if len(neg) >= N_NEG: break
        f = int(rng.integers(g, max(g+1, hi)))
        if all(abs(f-p) > SEG_FR for p in pc): neg.append(crop(af, f))
    if not neg:   # nothing clean left to contrast against; don't crash on an empty pool
        neg = [np.full((N_MELS, SEG_FR), af.min(), dtype="float32")]
    m = copy.deepcopy(enc).to(device); h = nn.Linear(EMB, 2).to(device)
    opt = optim.Adam(list(m.parameters())+list(h.parameters()), FT_LR)
    ce = nn.CrossEntropyLoss(); m.train(); h.train(); hf = FT_BATCH//2
    for _ in range(FT_STEPS):
        xs = [specaug(pos[int(rng.integers(len(pos)))], rng) for _ in range(hf)]
        xs += [neg[int(rng.integers(len(neg)))] for _ in range(hf)]
        xb = torch.tensor(np.stack(xs)).to(device)
        yb = torch.tensor([1]*hf+[0]*hf).to(device)
        loss = ce(h(m(xb)), yb); opt.zero_grad(); loss.backward(); opt.step()
    m.eval(); h.eval()
    fr = list(range(g, max(g+1, sf.shape[1]-g), STRIDE)); pb = np.empty(len(fr), dtype="float32")
    with torch.no_grad():
        for i in range(0, len(fr), 256):
            b = torch.tensor(np.stack([crop(sf, f) for f in fr[i:i+256]])).to(device)
            p = torch.softmax(h(m(b)), 1)[:, 1]; pb[i:i+len(p)] = p.cpu().numpy()
    return stitch(fr, pb)

# ---------------------------------------------- detection carrying BOTH scores
def detect_pv(rec, rng):
    """-> [(start, end, p, v)]  p = stage-1 probability, v = Perch verification.
    Perch sees a real 5s window of THIS recording centred on the event."""
    ev = stage1(encoder, rec, support, rng)
    if not ev: return []
    centres = [(s + e) / 2 for s, e, _ in ev]
    emb = l2n(perch_embed_ctx(to32k(rec), centres))
    v = perch_clf.predict_proba(emb)[:, 1]
    return [(s, e, p, float(vi)) for (s, e, p), vi in zip(ev, v)]

# Fusion rules. p alone is the stage-1 baseline; v alone makes stage-1 a pure
# proposal generator and lets Perch do all the ranking. The exponents trade off
# how much each stage is trusted -- sqrt(v) softens the veto (protects recall),
# sqrt(p) leans on Perch (protects precision).
FUSIONS = {
    "stage-1 only  (p)":  lambda p, v: p,
    "perch only    (v)":  lambda p, v: v,
    "p * v":              lambda p, v: p * v,
    "p * sqrt(v)":        lambda p, v: p * np.sqrt(v),
    "sqrt(p) * v":        lambda p, v: np.sqrt(p) * v,
    "geo mean sqrt(p*v)": lambda p, v: np.sqrt(p * v),
}

# ------------------------------------------------------------------- metrics
def match(events, truth, iou_thresh):
    """Greedy one-to-one matching, highest IoU first. -> per-EVENT (score, hit)."""
    pairs = sorted(((iou(e[:2], t), i, j) for i, e in enumerate(events)
                    for j, t in enumerate(truth)), reverse=True)
    used_e, used_t = set(), set()
    for score, i, j in pairs:
        if score <= iou_thresh: break
        if i in used_e or j in used_t: continue
        used_e.add(i); used_t.add(j)
    return [(e[2], i in used_e) for i, e in enumerate(events)]

def match_truth(events, truth, iou_thresh):
    """Same matching, but -> per-TRUTH matched score (0.0 if missed).
    Needed for recall-by-SNR: ablation #1 rejected Perch as 'blind to faint calls',
    so the verifier is expected to fail hardest on the quietest planted calls. This
    is the measurement that tests that prediction instead of assuming it."""
    pairs = sorted(((iou(e[:2], t), i, j) for i, e in enumerate(events)
                    for j, t in enumerate(truth)), reverse=True)
    ue, ut, out = set(), set(), [0.0] * len(truth)
    for score, i, j in pairs:
        if score <= iou_thresh: break
        if i in ue or j in ut: continue
        ue.add(i); ut.add(j); out[j] = events[i][2]
    return out

def fp_per_tp(scored, n_truth, target_recall=0.5):
    """False positives per true positive at a fixed recall -- the currency PRD v2.0
    §2.3/RQ3 cares about (analyst verification burden), not F1."""
    arr = sorted(scored, key=lambda x: -x[0]); tp = fp = 0
    for score, hit in arr:
        if hit: tp += 1
        else:   fp += 1
        if tp / n_truth >= target_recall:
            return fp / tp if tp else float('inf')
    return float('inf')

def summarise(scored, n_truth):
    """Exact PR curve over the observed scores -- no fixed threshold grid, which
    would misread the compressed range of the fused scores."""
    if not scored or not n_truth:
        return dict(ap=0.0, f1=0.0, p=0.0, r=0.0, thr=1.0, n=len(scored))
    arr = sorted(scored, key=lambda x: -x[0])
    tp = fp = 0; ap = 0.0; prev_r = 0.0; best = (0.0, 0.0, 0.0, 1.0)
    for score, hit in arr:
        if hit: tp += 1
        else:   fp += 1
        pr = tp / (tp + fp); rc = tp / n_truth
        ap += pr * (rc - prev_r); prev_r = rc
        f1 = 2*pr*rc/(pr+rc) if pr + rc else 0.0
        if f1 > best[0]: best = (f1, pr, rc, score)
    return dict(ap=ap, f1=best[0], p=best[1], r=best[2], thr=best[3], n=len(scored))

# --------------------------------------- train stage-1 encoder (target excluded)
rng = np.random.default_rng(0)
sps = [s for s in meta.primary_label.unique() if s != "whbsho3"]; rng.shuffle(sps); bank = {}
for sp in sps:
    if len(bank) >= 40: break
    segs = []
    for f in meta[meta.primary_label == sp]["filename"].tolist()[:12]:
        try:
            y = ld(os.path.join(AUD, f), 15); segs.append(crop(pcen(y), loud_off(y)//HOP + SEG_FR//2))
        except Exception: pass
    if len(segs) >= 8: bank[sp] = segs
print("training stage-1 encoder on", len(bank), "species...")
torch.manual_seed(0); encoder = train_enc(bank, np.random.default_rng(0))

# -------------------------------- PASS 1: F1 / AP on planted held-out target calls
def eval_pass(seed=7):
    r = np.random.default_rng(seed); cache = []
    for bed in eval_beds:
        y = bed.copy(); L = len(y); spans, planted, snrs = [], [], []
        for k in range(CALLS_PER_BED):
            snr = SNR_LEVELS[k % len(SNR_LEVELS)]
            placed = None
            for _ in range(80):
                pos = int(r.integers(0, L-CLEN))
                if any(pos < e+int(0.4*SR) and pos+CLEN > s-int(0.4*SR) for s, e in spans): continue
                placed = pos; break
            if placed is None: continue
            c = target_pool[int(r.integers(len(target_pool)))][:CLEN]
            if len(c) < CLEN: c = np.pad(c, (0, CLEN-len(c)))
            g = rms(y[placed:placed+CLEN])*(10**(snr/20))/(rms(c)+1e-9)
            y[placed:placed+CLEN] += c.astype("float32")*g
            spans.append((placed, placed+CLEN))
            planted.append((placed/SR, (placed+CLEN)/SR)); snrs.append(snr)
        cache.append((detect_pv(y, r), planted, snrs))
    return cache

# --------------------------- PASS 2: specificity control (target vs. decoys, equal loudness)
def spec_pass(seed=7):
    r = np.random.default_rng(seed); cache = []
    for bed in eval_beds:
        y = bed.copy(); L = len(y); spans, items = [], []
        plan = [("target", target_pool)]*6
        for n_, p_ in decoy_pools.items(): plan += [(n_, p_)]*3
        for kind, pool in plan:
            placed = None
            for _ in range(80):
                pos = int(r.integers(0, L-CLEN))
                if any(pos < e+int(0.4*SR) and pos+CLEN > s-int(0.4*SR) for s, e in spans): continue
                placed = pos; break
            if placed is None or not pool: continue
            c = pool[int(r.integers(len(pool)))][:CLEN]
            if len(c) < CLEN: c = np.pad(c, (0, CLEN-len(c)))
            g = rms(y[placed:placed+CLEN])*(10**(6.0/20))/(rms(c)+1e-9)
            y[placed:placed+CLEN] += c.astype("float32")*g
            spans.append((placed, placed+CLEN))
            items.append((kind, (placed/SR, (placed+CLEN)/SR)))
        cache.append((detect_pv(y, r), items))
    return cache

# PRD v2.0 §7.2: N>=5 seeds, matched-pair, "treat |delta/SE| < ~2 as not evidence".
# 3 is the 2-day compromise; raise to 5 if there's runtime. Cost is linear in len(SEEDS).
SEEDS = (7, 8, 9)
eval_caches, spec_caches = [], []
for _sd in SEEDS:
    print(f"\nseed {_sd}: pass 1/2 (F1/AP)...");        eval_caches.append(eval_pass(_sd))
    print(f"seed {_sd}: pass 2/2 (specificity)...");    spec_caches.append(spec_pass(_sd))

# ------------------------------------------------------ score every fusion rule
def score_eval(fuse, eval_cache):
    scored, n_truth, by_snr = [], 0, []
    for dets, planted, snrs in eval_cache:
        ev = [(s, e, float(fuse(p, v))) for s, e, p, v in dets]
        scored += match(ev, planted, IOU_THR); n_truth += len(planted)
        by_snr += list(zip(snrs, match_truth(ev, planted, IOU_THR)))
    m = summarise(scored, n_truth)
    m["fp_per_tp@R50"] = fp_per_tp(scored, n_truth, 0.5)
    m["recall_by_snr"] = {s: float(np.mean([sc >= m["thr"] for ss, sc in by_snr if ss == s]))
                          for s in SNR_LEVELS}
    return m

def _auc(pos, neg):
    """P(random target scores above random decoy), ties at 0.5. Threshold-FREE.
    0.50 = cannot tell target from decoy at all (pure any-bird detector);
    1.00 = perfect species discrimination. Use this to compare fusion rules --
    the hit-rate ratio below is measured at the best-F1 threshold, which is not
    binding on the +6 dB planted calls and therefore cannot separate the rules."""
    pos, neg = np.asarray(pos, float), np.asarray(neg, float)
    if not len(pos) or not len(neg): return float('nan')
    gt = float((pos[:, None] > neg[None, :]).sum())
    eq = float((pos[:, None] == neg[None, :]).sum())
    return (gt + 0.5 * eq) / (len(pos) * len(neg))

def score_spec(fuse, thr, spec_cache):
    """Hit rates at the deployed operating point, plus a threshold-free AUC."""
    tg, dc = [], {k: [] for k in decoy_pools}
    for dets, items in spec_cache:
        ev = [(s, e, float(fuse(p, v))) for s, e, p, v in dets]
        for kind, span in items:
            best = max((iou(e[:2], span), e[2]) for e in ev) if ev else (0.0, 0.0)
            sc = best[1] if best[0] > IOU_THR else 0.0
            (tg if kind == "target" else dc[kind]).append(sc)
    t = float(np.mean([s >= thr for s in tg])) if tg else 0.0
    d = {k: float(np.mean([s >= thr for s in v])) for k, v in dc.items() if v}
    md = float(np.mean(list(d.values()))) if d else 0.0
    all_dc = [s for v in dc.values() for s in v]
    return t, md, (md/t if t else float('nan')), d, _auc(tg, all_dc)

per_seed = {name: [] for name in FUSIONS}
for ec, sc in zip(eval_caches, spec_caches):
    for name, fuse in FUSIONS.items():
        m = score_eval(fuse, ec)
        t, md, ratio, per, auc = score_spec(fuse, m["thr"], sc)
        per_seed[name].append(dict(ap=m["ap"], f1=m["f1"], p=m["p"], r=m["r"],
                                   fptp=m["fp_per_tp@R50"], auc=auc, ratio=ratio,
                                   snr=m["recall_by_snr"], per=per))

def agg(name, key):
    v = np.array([d[key] for d in per_seed[name]], dtype=float)
    return float(v.mean()), (float(v.std(ddof=1)) if len(v) > 1 else 0.0)

n_planted = sum(len(p) for _, p, _ in eval_caches[0])
print(f"\n{'='*92}\nF1 vs SPECIFICITY -- {len(eval_beds)} beds, {n_planted} planted target calls/seed,"
      f" strict IoU>={IOU_THR}\nN={len(SEEDS)} seeds {SEEDS}, mean +/- sd."
      f"  Within a seed the rules share identical detections (paired).\n{'='*92}")
print(f"{'fusion rule':<21} {'AP':>12} {'bestF1':>12} {'prec':>6} {'rec':>6} {'FP/TP':>6} "
      f"{'|':>2} {'ratio':>6} {'specAUC':>13}")
print("-"*92)
for name in FUSIONS:
    ap, aps = agg(name, "ap"); f1, f1s = agg(name, "f1"); auc, aucs = agg(name, "auc")
    print(f"{name:<21} {ap:6.3f}±{aps:5.3f} {f1:6.3f}±{f1s:5.3f} {agg(name,'p')[0]:6.1%} "
          f"{agg(name,'r')[0]:6.1%} {agg(name,'fptp')[0]:6.1f} {'|':>2} "
          f"{agg(name,'ratio')[0]:6.2f} {auc:6.3f}±{aucs:5.3f}")
print(" specAUC threshold-free: 0.50 = target indistinguishable from decoys, 1.00 = perfect.")

# Does Perch verification cost us the FAINT calls? argus.py ablation #1 rejected Perch
# as "blind to faint calls" -- if that still holds, the two-stage rows collapse at the
# bottom SNRs while stage-1 holds up. This is the falsifiable version of that claim.
print(f"\nrecall by planted-call SNR (mean over seeds, each variant at its own best-F1 threshold)")
print(f"{'fusion rule':<21}" + "".join(f"{s:+4d}dB" for s in SNR_LEVELS))
print("-"*(21 + 6*len(SNR_LEVELS)))
for name in FUSIONS:
    print(f"{name:<21}" + "".join(
        f"{np.mean([d['snr'][s] for d in per_seed[name]]):5.0%} " for s in SNR_LEVELS))

# ---- the decisive comparison, matched-pair per PRD v2.0 §7.2 ----
BASE, CAND = "stage-1 only  (p)", "perch only    (v)"
def paired(key):
    d = np.array([c[key] - b[key] for b, c in zip(per_seed[BASE], per_seed[CAND])], dtype=float)
    m = float(d.mean())
    se = float(d.std(ddof=1) / np.sqrt(len(d))) if len(d) > 1 else float('nan')
    return m, se, (m / se if se else float('nan'))

print(f"\n{'='*92}\nPAIRED: {CAND.strip()} vs {BASE.strip()}  (same detections, N={len(SEEDS)} seeds)\n{'='*92}")
print(f"{'metric':<14}{'delta':>9}{'SE':>8}{'delta/SE':>10}   verdict")
for key, label, good in (("f1", "best F1", "+"), ("ap", "AP", "+"),
                         ("auc", "specificity AUC", "+"), ("fptp", "FP per TP", "-")):
    m, se, t = paired(key)
    verdict = ("not evidence (|d/SE|<2)" if not np.isfinite(t) or abs(t) < 2
               else ("supports Perch" if (t > 0) == (good == "+") else "favours stage-1"))
    print(f"{label:<14}{m:>9.3f}{se:>8.3f}{t:>10.2f}   {verdict}")
print("\nPRD v2.0 §7.2: |delta/SE| < ~2 is NOT evidence. §7.3: one target species is not")
print("a validated finding -- the multi-species analogue of the 12-clip rule is still owed.")


device: cuda
training stage-1 encoder on 40 species...
    enc ep 250/1000 loss 1.081
    enc ep 500/1000 loss 1.045
    enc ep 750/1000 loss 0.803
    enc ep 1000/1000 loss 0.644

seed 7: pass 1/2 (F1/AP)...
seed 7: pass 2/2 (specificity)...

seed 8: pass 1/2 (F1/AP)...


In [ ]:
# ===== CELL 3 / DEMO SERVER -- run AFTER cell 1 (setup) and cell 2 (arguswala.py) =====
# Serves the detector over HTTP and opens a public tunnel so a website can call it.
# Reuses from earlier cells: encoder, support, target_pool, eval_beds, detect_pv,
#   SR, CLEN, SNR_LEVELS, rms, IOU_THR, iou
#
# While building the demo, set SEEDS = (7,) in cell 2 -- you only need `encoder` and
# `detect_pv` to exist, not the full research sweep.
#
# WHAT THE DEMO SHOWS (and why it's built this way):
# The unlabeled soundscapes are real Western Ghats field audio, but there's no
# guarantee the target species is in any given one -- so "run it on raw forest audio"
# would often show nothing, which demos badly AND can't be verified by a judge.
# Instead each scenario plants a known number of real target calls at known times and
# known loudnesses (+9 dB down to -6 dB). The page shows ground truth alongside the
# detections, so a judge can see hits, misses, and false alarms, and can LISTEN to any
# of them. That is both more honest and more convincing than an unverifiable clip.
import os, io, re, json, time, base64, threading, subprocess
import numpy as np, librosa, librosa.display
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import soundfile as sf

for _pkg, _mod in (("fastapi", "fastapi"), ("uvicorn", "uvicorn"),
                   ("nest_asyncio", "nest_asyncio"), ("python-multipart", "multipart")):
    try: __import__(_mod)          # python-multipart is required for file uploads
    except ImportError: subprocess.run(["pip", "install", "-q", _pkg], check=True)

import nest_asyncio, uvicorn
from fastapi import FastAPI, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

PORT = 8000
DEMO_SEC = 120.0                      # 60 s was too short for the adaptation step to work
DEMO_SNRS = (12, 9, 6, 3, 0, -3)      # loud -> faint, so the hard cases stay visible
N_SCENARIOS = 3
# Fusion is chosen live on the site (Stage 1 / Verifier / p*v -- see FUSIONS_DEMO below),
# not fixed here. The site defaults to "pv": Perch-alone has the best precision on paper,
# but its recall on the LOUDEST planted calls is oddly low (55% at +9 dB vs 80% for
# stage-1, in both runs) -- missing a call the judge can plainly hear looks far worse in
# a live demo than one extra false alarm.

# ------------------------------------------------------------------ audio helpers
def wav_b64(y, sr=SR):
    buf = io.BytesIO()
    sf.write(buf, np.clip(np.asarray(y, dtype="float32"), -1, 1), sr, format="WAV", subtype="PCM_16")
    return base64.b64encode(buf.getvalue()).decode()

def spec_b64(y, sr=SR):
    """Spectrogram PNG with NO axes or padding, so the image spans exactly
    0 -> duration edge to edge. That lets the web page position detection and
    ground-truth markers by percentage and have them line up precisely, and keeps
    them clickable instead of burned into the picture."""
    S = librosa.amplitude_to_db(np.abs(librosa.stft(y, n_fft=1024, hop_length=256)), ref=np.max)
    keep = librosa.fft_frequencies(sr=sr, n_fft=1024) <= 11000
    fig = plt.figure(figsize=(16, 4.2), dpi=100)
    ax = fig.add_axes([0, 0, 1, 1]); ax.axis("off")
    ax.imshow(S[keep], origin="lower", aspect="auto", cmap="magma", vmin=-70, vmax=0)
    buf = io.BytesIO()
    fig.savefig(buf, format="png", transparent=True, pad_inches=0); plt.close(fig)
    return base64.b64encode(buf.getvalue()).decode()

# ------------------------------------------------------- build the demo scenarios
def build_scenario(bed, seed):
    """60 s of real forest audio + one known target call per SNR level."""
    r = np.random.default_rng(seed)
    n = int(DEMO_SEC * SR)
    off = int(r.integers(0, max(1, len(bed) - n)))
    y = bed[off:off + n].copy()
    spans, truth = [], []
    for snr in DEMO_SNRS:
        placed = None
        for _ in range(120):
            pos = int(r.integers(0, len(y) - CLEN))
            if any(pos < e + int(1.2*SR) and pos + CLEN > s - int(1.2*SR) for s, e in spans): continue
            placed = pos; break
        if placed is None: continue
        c = target_pool[int(r.integers(len(target_pool)))][:CLEN]
        if len(c) < CLEN: c = np.pad(c, (0, CLEN - len(c)))
        g = rms(y[placed:placed+CLEN]) * (10 ** (snr/20)) / (rms(c) + 1e-9)
        y[placed:placed+CLEN] += c.astype("float32") * g
        spans.append((placed, placed + CLEN))
        truth.append({"start": placed/SR, "end": (placed+CLEN)/SR, "snr": int(snr)})
    peak = float(np.max(np.abs(y))) or 1.0
    if peak > 0.99: y = y * (0.99 / peak)
    return y.astype("float32"), sorted(truth, key=lambda t: t["start"])

print("building demo scenarios...")
SCENARIOS = []
for i in range(N_SCENARIOS):
    y, truth = build_scenario(eval_beds[i % len(eval_beds)], seed=100 + i)
    SCENARIOS.append({"id": i, "name": f"Western Ghats soundscape #{i+1}",
                      "seconds": round(len(y)/SR, 1), "audio": y, "truth": truth})
    print(f"  scenario {i+1}: {len(truth)} planted calls, {len(y)/SR:.0f}s")

SUPPORT_PAYLOAD = {
    "count": len(support),
    "clips": [wav_b64(w) for w in support[:5]],
    "note": "The only labelled examples the model is given. Everything else is found unaided.",
}

# ------------------------------------------------------------------- detection
# All fusion scoring happens client-side (the site has the raw p/v for every candidate
# and picks by whichever tab is selected) -- this dict exists only so the server can
# calibrate a matching threshold per rule. Keys match the site's data-f="p"/"v"/"pv"
# exactly. A single threshold calibrated for one rule would be silently wrong for the
# others: p*v scores are much smaller than either factor alone, so reusing one number
# across all three would flood or empty the demo depending on which way you switched.
FUSIONS_DEMO = {"p": lambda p, v: p, "v": lambda p, v: v, "pv": lambda p, v: p * v}

# Does the stage1() currently in memory already scale injections by clip length?
# If cell 2 is an older copy that hardcodes 45, we patch the global below; if it's the
# fixed version, we must NOT, or the density gets halved twice.
try:
    import inspect
    _STAGE1_SCALES = "n_inj" in inspect.getsource(stage1)
except Exception:
    _STAGE1_SCALES = False
print("stage1 scales injections by clip length:", _STAGE1_SCALES,
      "" if _STAGE1_SCALES else "-> demo will patch N_INJECT itself")

_RAW = {}
def raw_detect(scen):
    """Unthresholded candidates for one scenario, cached. Detection is the slow part;
    choosing a cutoff afterwards is free."""
    global N_INJECT
    if scen["id"] in _RAW: return _RAW[scen["id"]]
    _saved = N_INJECT
    if not _STAGE1_SCALES:
        N_INJECT = max(8, int(round(45 * len(scen["audio"]) / (240.0 * SR))))
    try:
        dets = detect_pv(scen["audio"], np.random.default_rng(0))
    finally:
        N_INJECT = _saved
    _RAW[scen["id"]] = dets
    return dets

def _match(events, truth_spans):
    pairs = sorted(((iou((e[0], e[1]), t), i, j)
                    for i, e in enumerate(events) for j, t in enumerate(truth_spans)), reverse=True)
    ue, ut = set(), set()
    for ov, i, j in pairs:
        if ov <= IOU_THR: break
        if i in ue or j in ut: continue
        ue.add(i); ut.add(j)
    return ue, ut

def _best_threshold(fn):
    """Sweep every observed score for fusion rule fn and take the best-F1 cutoff --
    the way the research harness does it -- instead of guessing a fixed number like 0.5,
    which is meaningless for a product of two probabilities (0.7*0.7=0.49, discarded)."""
    allscores, per = [], []
    for s in SCENARIOS:
        ev = [(a, b, float(fn(p, v))) for a, b, p, v in raw_detect(s)]
        per.append((ev, [(t["start"], t["end"]) for t in s["truth"]]))
        allscores += [e[2] for e in ev]
    if not allscores:
        return 0.0, -1.0, 0
    best, best_f1 = 0.0, -1.0
    for thr in sorted(set(round(x, 4) for x in allscores)):
        tp = fp = nt = 0
        for ev, truth in per:
            keep = [e for e in ev if e[2] >= thr]
            ue, ut = _match(keep, truth)
            tp += len(ut); fp += len(keep) - len(ue); nt += len(truth)
        pr = tp/(tp+fp) if tp+fp else 0.0; rc = tp/nt if nt else 0.0
        f1 = 2*pr*rc/(pr+rc) if pr+rc else 0.0
        if f1 > best_f1: best_f1, best = f1, thr
    return best, best_f1, len(allscores)

def calibrate_all():
    """One best-F1 threshold PER fusion rule (p / v / pv), so switching the fusion tab
    on the site always lands on a sensibly-calibrated cutoff instead of reusing whatever
    was calibrated for a different rule."""
    out = {}
    for key, fn in FUSIONS_DEMO.items():
        thr, f1, n = _best_threshold(fn)
        out[key] = thr
        if n == 0:
            print(f"  calibrate[{key}]: stage 1 proposed NOTHING -- a real failure, not a threshold problem")
        else:
            print(f"  calibrate[{key}]: {n} candidates -> threshold {thr:.3f} (F1 {f1:.3f})")
    return out

print("\nrunning detection on the demo clips and calibrating one threshold per fusion rule...")
AUTO_THR = calibrate_all()   # dict: {"p": ..., "v": ..., "pv": ...}

def run_detection(scen):
    """Returns EVERY candidate with both stage scores, unthresholded. The page then
    filters, matches and scores in the browser -- so moving a threshold slider or
    switching the fusion rule is instant instead of a round trip to the GPU."""
    t0 = time.time()
    dets = raw_detect(scen)
    cands = []
    for s, e, p, v in dets:
        a, b = int(s*SR), int(e*SR)
        clip = scen["audio"][max(0, a):min(len(scen["audio"]), b)]
        if len(clip) < int(0.4*SR): clip = np.pad(clip, (0, int(0.4*SR) - len(clip)))
        cands.append({"start": round(float(s), 2), "end": round(float(e), 2),
                      "p": round(float(p), 4), "v": round(float(v), 4),
                      "audio": wav_b64(clip)})
    cands.sort(key=lambda c: c["start"])
    return {"candidates": cands, "truth": scen["truth"],
            "thresholds": AUTO_THR, "elapsed": round(time.time() - t0, 1)}

def _pca2(X):
    Xc = np.asarray(X, dtype="float64"); Xc = Xc - Xc.mean(0)
    _, _, Vt = np.linalg.svd(Xc, full_matrices=False)
    return (Xc @ Vt[:2].T)

def analysis_payload():
    """Real measured internals, not decoration: where the species sit in the verifier's
    embedding space, how every candidate scored on each stage, and the research table
    from cell 2 if it was run in this session. (Route registered below, once `app` exists.)"""
    out = {"target": TARGET}

    # --- which species the target is actually confusable with ---
    try:
        out["decoys"] = [{"name": sp, "sim": round(cos(species_emb[sp], PROTO), 3),
                          "calls": len(decoy_pools[sp])} for sp in DECOYS]
    except Exception:
        out["decoys"] = []

    # --- 2D projection of Perch embedding space: target vs its nearest confusables ---
    try:
        groups, mats = [], []
        groups.append(("target", len(support)));           mats.append(l2n(perch_embed(support)))
        if target_pool:
            groups.append(("target (held out)", len(target_pool)))
            mats.append(l2n(perch_embed(target_pool)))
        for sp in DECOYS:
            pool = decoy_pools[sp][:20]
            groups.append((sp, len(pool))); mats.append(l2n(perch_embed(pool)))
        P = _pca2(np.concatenate(mats))
        rng_ = 0; proj = {}
        for name, n in groups:
            proj[name] = [[round(float(x), 3), round(float(y), 3)] for x, y in P[rng_:rng_+n]]
            rng_ += n
        out["projection"] = proj
    except Exception as ex:
        out["projection"] = {}; out["projection_error"] = f"{type(ex).__name__}: {ex}"

    # --- every demo candidate as (stage-1 score, verifier score, was it real) ---
    try:
        pts = []
        for s in SCENARIOS:
            truth = [(t["start"], t["end"]) for t in s["truth"]]
            ev = [(a, b, 1.0) for a, b, _, _ in raw_detect(s)]
            ue, _ = _match([(e[0], e[1]) for e in ev], truth)
            for i, (a, b, p, v) in enumerate(raw_detect(s)):
                pts.append({"p": round(float(p), 4), "v": round(float(v), 4),
                            "hit": i in ue, "scenario": s["id"]})
        out["candidates"] = pts
    except Exception:
        out["candidates"] = []

    # --- the research table, straight out of cell 2 if it ran in this session ---
    try:
        tbl = []
        for name in per_seed:
            d = per_seed[name]
            m = lambda k: round(float(np.mean([x[k] for x in d])), 3)
            sd = lambda k: round(float(np.std([x[k] for x in d], ddof=1)), 3) if len(d) > 1 else 0.0
            tbl.append({"rule": name.strip(), "ap": m("ap"), "ap_sd": sd("ap"),
                        "f1": m("f1"), "f1_sd": sd("f1"), "prec": m("p"), "rec": m("r"),
                        "fptp": m("fptp"), "auc": m("auc"), "auc_sd": sd("auc"),
                        "ratio": m("ratio"),
                        "snr": {str(k): round(float(np.mean([x["snr"][k] for x in d])), 3)
                                for k in SNR_LEVELS}})
        out["research"] = {"table": tbl, "seeds": len(per_seed[list(per_seed)[0]]),
                           "snr_levels": list(SNR_LEVELS)}
    except Exception:
        out["research"] = None
    return out

# ------------------------------------------------------------------------ API
app = FastAPI(title="ARGUS")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

@app.get("/api/analysis")
def analysis(): return analysis_payload()

class DetectReq(BaseModel):
    scenario: int = 0

@app.get("/api/health")
def health():
    return {"ok": True, "target": TARGET, "device": str(device), "thresholds": AUTO_THR}

@app.get("/api/scenarios")
def scenarios():
    return {"target": TARGET, "support": SUPPORT_PAYLOAD,
            "scenarios": [{"id": s["id"], "name": s["name"], "seconds": s["seconds"],
                           "planted": len(s["truth"])} for s in SCENARIOS]}

@app.get("/api/clip/{sid}")
def clip(sid: int):
    s = SCENARIOS[sid]
    return {"id": sid, "audio": wav_b64(s["audio"]), "spectrogram": spec_b64(s["audio"]),
            "seconds": s["seconds"]}

@app.post("/api/detect")
def detect(req: DetectReq):
    return run_detection(SCENARIOS[req.scenario])

MAX_UPLOAD_SEC = 180.0

@app.post("/api/detect_upload")
async def detect_upload(file: UploadFile = File(...)):
    """Run the detector on audio handed to it live -- a judge's phone recording, a
    field clip, anything librosa can open. No ground truth here, so the response
    carries detections only; there is nothing to score against."""
    global N_INJECT
    t0 = time.time()
    raw = await file.read()
    try:
        y, _ = librosa.load(io.BytesIO(raw), sr=SR, mono=True, duration=MAX_UPLOAD_SEC)
    except Exception as ex:
        return {"error": f"could not read that audio ({type(ex).__name__}). Try WAV, OGG, FLAC or MP3."}
    y = y.astype("float32")
    if len(y) < 3 * SR:
        return {"error": "clip is too short -- needs at least ~3 seconds to adapt to."}
    _saved = N_INJECT
    if not _STAGE1_SCALES:
        N_INJECT = max(8, int(round(45 * len(y) / (240.0 * SR))))
    try:
        dets = detect_pv(y, np.random.default_rng(0))
    finally:
        N_INJECT = _saved
    events = []
    for s, e, p, v in dets:
        a, b = int(s*SR), int(e*SR)
        clip = y[max(0, a):min(len(y), b)]
        if len(clip) < int(0.4*SR): clip = np.pad(clip, (0, int(0.4*SR) - len(clip)))
        events.append({"start": round(float(s), 2), "end": round(float(e), 2),
                       "p": round(float(p), 4), "v": round(float(v), 4),
                       "audio": wav_b64(clip)})
    events.sort(key=lambda e: e["start"])
    return {"filename": file.filename, "seconds": round(len(y)/SR, 1),
            "audio": wav_b64(y), "spectrogram": spec_b64(y), "candidates": events,
            "thresholds": AUTO_THR, "elapsed": round(time.time() - t0, 1)}

@app.get("/api/snapshot")
def snapshot():
    """Everything the page needs, precomputed -- save this to disk and the site runs
    with no backend at all. This is the expo insurance policy: if the venue wifi
    blocks the tunnel, the demo is identical from the audience's side."""
    out = {"target": TARGET, "support": SUPPORT_PAYLOAD,
           "thresholds": AUTO_THR, "analysis": analysis_payload(), "scenarios": []}
    for s in SCENARIOS:
        res = run_detection(s)
        out["scenarios"].append({"id": s["id"], "name": s["name"], "seconds": s["seconds"],
                                 "audio": wav_b64(s["audio"]), "spectrogram": spec_b64(s["audio"]),
                                 **res})
    return out

# --------------------------------------------------------------- serve + tunnel
nest_asyncio.apply()

def _free_port(start=PORT, tries=20):
    """Re-running this cell leaves the previous server still bound to its port; without
    this, the new app silently never gets served and the tunnel reaches the OLD code."""
    import socket
    for p in range(start, start + tries):
        with socket.socket() as s:
            try: s.bind(("0.0.0.0", p)); return p
            except OSError: continue
    return start

PORT = _free_port()
_cfg = uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning")
threading.Thread(target=uvicorn.Server(_cfg).run, daemon=True).start()
time.sleep(3)
print(f"local server up on :{PORT}")

# ---- STEP 1: the offline snapshot FIRST. It needs no internet, so it must not be
# ---- downstream of the tunnel -- a fallback that fails when the primary fails is useless.
SNAP_PATH = "/kaggle/working/argus_snapshot.json"
print("\nbuilding offline snapshot (this is the one that has to work)...")
try:
    with open(SNAP_PATH, "w") as fh: json.dump(snapshot(), fh)   # carries all 3 calibrated thresholds
    print(f"  OK -> {SNAP_PATH}  ({os.path.getsize(SNAP_PATH)/1e6:.1f} MB)")
    print("  Download it from the Output panel on the right. The website runs entirely")
    print("  from this file with no network at all -- use 'Load offline snapshot'.")
except Exception as ex:
    print("  snapshot FAILED:", type(ex).__name__, ex)

# ---- STEP 2: the public tunnel. Optional, and never fatal.
def _fetch_cloudflared():
    url = ("https://github.com/cloudflare/cloudflared/releases/latest/download/"
           "cloudflared-linux-amd64")
    for cmd in (["curl", "-sL", "-o", "cloudflared", url],
                ["wget", "-q", "-O", "cloudflared", url]):
        try:
            subprocess.run(cmd, check=True, timeout=180)
            if os.path.getsize("cloudflared") > 1_000_000: return True
        except Exception: pass
    try:
        import urllib.request
        urllib.request.urlretrieve(url, "cloudflared")
        return os.path.getsize("cloudflared") > 1_000_000
    except Exception:
        return False

def open_tunnel(port=PORT):
    if not (os.path.exists("cloudflared") and os.path.getsize("cloudflared") > 1_000_000):
        if not _fetch_cloudflared(): return None, None
    os.chmod("cloudflared", 0o755)
    proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", f"http://localhost:{port}",
                             "--no-autoupdate"], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    t0 = time.time()
    for line in proc.stdout:
        m = re.search(r"https://[-\w]+\.trycloudflare\.com", line)
        if m: return proc, m.group(0)
        if time.time() - t0 > 90: break
    return proc, None

_proc, PUBLIC_URL = open_tunnel()
print("\n" + "="*70)
if PUBLIC_URL:
    print(f"  PUBLIC URL:  {PUBLIC_URL}")
    print("  Paste this into the website's box and press Connect.")
else:
    print("  NO TUNNEL — almost always means Internet is OFF for this notebook.")
    print("  Fix: right sidebar -> Settings -> Internet -> On, then re-run this cell.")
    print("  (Needs a phone-verified Kaggle account. Nothing else in the project needs")
    print("   internet, which is why this is the first time it has come up.)")
    print(f"\n  You can still demo right now: use {SNAP_PATH} with 'Load offline snapshot'.")
print("="*70)


In [ ]:
import json, os
p = "/kaggle/working/argus_snapshot.json"
with open(p, "w") as fh: json.dump(snapshot(), fh)
print(f"{os.path.getsize(p)/1e6:.1f} MB")